In [31]:
import numpy as np
import open3d as o3d
from pathlib import Path
import torch
import pickle
import os


In [32]:
UHM_DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train")


In [33]:
print(os.path.exists(UHM_DATASET_DIR))
print(os.path.isdir(UHM_DATASET_DIR))

True
True


In [34]:
DATASET_DIR = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/faces")
os.makedirs(DATASET_DIR, exist_ok=True)

In [35]:
print(os.path.exists(DATASET_DIR))
print(os.path.isdir(DATASET_DIR))

True
True


In [36]:
all_files = list(UHM_DATASET_DIR.glob("*.ply"))

In [37]:
pca_data = torch.load("pca_basis_all.pth")
all_gt_z = pca_data['gt_z']
all_sorted_filenames = sorted([f.name for f in UHM_DATASET_DIR.glob("*.ply")])
name_to_idx = {name: idx for idx, name in enumerate(all_sorted_filenames)}
print(f"Loaded GT Z tensor of shape: {all_gt_z.shape}")

Loaded GT Z tensor of shape: torch.Size([32, 8000, 100])


In [38]:
RANDOM_SEED = 42

In [39]:
np.random.seed(RANDOM_SEED)

In [40]:
N_SAMPLES = 2500  # 80/10/10 split → 2000 train, 250 val, 250 test

In [41]:
random_sample = np.random.choice(all_files, size=N_SAMPLES, replace=False)

In [42]:
random_sample

array([WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3478.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/3890.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/2871.ply'),
       ...,
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/784.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/6136.ply'),
       WindowsPath('C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train/8990.ply')],
      dtype=object)

In [43]:
rng = np.random.default_rng(RANDOM_SEED)

In [44]:
"""
def generate_random_view(
    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9
) -> np.ndarray:

    center = pcd_points.mean(axis=0)
    extent_y = np.ptp(pts[:, 1])
    extent_z = np.ptp(pts[:, 2])
    height = extent_y * plane_height_factor
    depth = extent_z * plane_depth_factor

    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)
    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])

    x_angle = rng.uniform(0, 2 * np.pi)
    y_angle = rng.uniform(0, 2 * np.pi)
    z_angle = rng.uniform(0, 2 * np.pi)
    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])
    plane.rotate(R_plane, center=center)

    initial_normal = np.array([1.0, 0.0, 0.0])
    rotated_normal = R_plane @ initial_normal
    rotated_normal /= np.linalg.norm(rotated_normal)

    vecs = pcd_points - center[np.newaxis, :]
    signed = vecs.dot(rotated_normal)
    side = rng.choice([1, -1])
    mask_side = (signed * side) > side_tol
    selected_pts = pcd_points[mask_side]
    orig_indices = np.nonzero(mask_side)[0]

    down_mask_bool = rng.choice([False, True], size=selected_pts.shape[0],
                                p=[1 - downsample_p, downsample_p])
    downsampled_pts = selected_pts[down_mask_bool]
    kept_indices = orig_indices[down_mask_bool]

    angles = rng.uniform(0, 2 * np.pi, size=3)
    R = o3d.geometry.get_rotation_matrix_from_xyz(angles)
    t = rng.uniform(-1.0, 1.0, size=3)
    transformed_points = (R @ downsampled_pts.T).T + t

    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4, dtype=float)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv

    print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((transformed_points, np.ones((transformed_points.shape[0], 1)))) @ T_inv.T)[:, :3] - pts[kept_indices])}")

    return transformed_points, R_inv, t_inv, plane, kept_indices
"""

'\ndef generate_random_view(\n    pcd_points: np.ndarray, plane_width=0.001, plane_height_factor=1.2, plane_depth_factor=1.2, downsample_p=0.5, side_tol=1e-9\n) -> np.ndarray:\n\n    center = pcd_points.mean(axis=0)\n    extent_y = np.ptp(pts[:, 1])\n    extent_z = np.ptp(pts[:, 2])\n    height = extent_y * plane_height_factor\n    depth = extent_z * plane_depth_factor\n\n    plane = o3d.geometry.TriangleMesh.create_box(width=plane_width, height=height, depth=depth)\n    plane.translate([center[0] - plane_width / 2, center[1] - height / 2, center[2] - depth / 2])\n\n    x_angle = rng.uniform(0, 2 * np.pi)\n    y_angle = rng.uniform(0, 2 * np.pi)\n    z_angle = rng.uniform(0, 2 * np.pi)\n    R_plane = o3d.geometry.get_rotation_matrix_from_xyz([x_angle, y_angle, z_angle])\n    plane.rotate(R_plane, center=center)\n\n    initial_normal = np.array([1.0, 0.0, 0.0])\n    rotated_normal = R_plane @ initial_normal\n    rotated_normal /= np.linalg.norm(rotated_normal)\n\n    vecs = pcd_points -

In [45]:
def generate_random_view(
    pcd_points: np.ndarray, 
    downsample_p=0.5, 
    camera_dist_factor=1.0, 
    **kwargs
):
    """
    Simulates a realistic view using Hidden Point Removal (HPR).
    kwargs absorbs legacy plane arguments (plane_width, plane_height_factor, etc.)
    """
    rng = np.random.default_rng()

    # 1. Convert numpy array to Open3D PointCloud
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pcd_points)

    # 2. Estimate dimensions and center
    min_bound = np.asarray(pcd.get_min_bound())
    max_bound = np.asarray(pcd.get_max_bound())
    diameter = np.linalg.norm(max_bound - min_bound)
    center = pcd_points.mean(axis=0)

    # 3. Generate a random camera viewpoint around the point cloud
    # Generate a random direction vector
    vec = rng.standard_normal(3)
    vec /= np.linalg.norm(vec)
    
    # Place camera at a distance from the center based on the diameter
    camera = center + vec * (diameter * camera_dist_factor)
    
    # 100x diameter is the standard recommended radius for Katz2007 HPR
    radius = diameter * 100.0 

    # 4. Apply Hidden Point Removal
    _, pt_map = pcd.hidden_point_removal(camera.tolist(), radius)
    
    # pt_map contains the indices of the visible points
    orig_indices = np.array(pt_map)
    selected_pts = pcd_points[orig_indices]

    # 5. Apply Downsampling (matching your original pipeline behavior)
    if downsample_p < 1.0:
        down_mask_bool = rng.choice([False, True], size=selected_pts.shape[0],
                                    p=[1 - downsample_p, downsample_p])
        downsampled_pts = selected_pts[down_mask_bool]
        kept_indices = orig_indices[down_mask_bool]
    else:
        downsampled_pts = selected_pts
        kept_indices = orig_indices

    # 6. Apply random rotation and translation
    angles = rng.uniform(0, 2 * np.pi, size=3)
    R = o3d.geometry.get_rotation_matrix_from_xyz(angles)
    t = rng.uniform(-1.0, 1.0, size=3)
    
    transformed_points = (R @ downsampled_pts.T).T + t

    # 7. Compute Inverse Transformations
    T = np.eye(4, dtype=float)
    T[:3, :3] = R
    T[:3, 3] = t

    R_inv = R.T
    t_inv = -R_inv @ t
    T_inv = np.eye(4, dtype=float)
    T_inv[:3, :3] = R_inv
    T_inv[:3, 3] = t_inv

    # Reprojection Error Check
    reproj_points = np.hstack((transformed_points, np.ones((transformed_points.shape[0], 1))))
    error = np.linalg.norm((reproj_points @ T_inv.T)[:, :3] - pcd_points[kept_indices])
    print(f"Reprojection error T_inv: {error}")

    # Note: Returning 'camera' coordinates instead of the old 'plane' mesh.
    return transformed_points, R_inv, t_inv, camera, kept_indices

In [46]:
random_file = random_sample[0]

In [47]:
pcd = o3d.io.read_point_cloud(str(random_file))
pcd.colors = o3d.utility.Vector3dVector(np.ones((len(pcd.points), 3)) * 0.5)
pts = np.asarray(pcd.points)

In [48]:
#view_points, view_r, view_t, view_plane, kept_indices = generate_random_view(pts)

view_points, view_r, view_t, view_camera, kept_indices = generate_random_view(pts)

Reprojection error T_inv: 1.011347595142623e-14


In [49]:
view_r.dtype

dtype('float64')

In [50]:
view_pcd = o3d.geometry.PointCloud()
view_pcd.points = o3d.utility.Vector3dVector(view_points)

In [51]:
#o3d.visualization.draw_plotly([view_pcd, view_plane, pcd])

o3d.visualization.draw_plotly([view_pcd, pcd])

In [52]:
T = np.eye(4, dtype=np.float64)
T[:3, :3] = view_r
T[:3, 3] = view_t

In [53]:
transformed_view = view_pcd.transform(T)

In [54]:
#o3d.visualization.draw_plotly([transformed_view, view_plane])

o3d.visualization.draw_plotly([transformed_view])

In [55]:
def build_metadata_dict(scene_name, pcd0_path, pcd1_path, pcd_morphed_path, gt_z_path, R, t, frag_id1, kept_indices):
    metadata = {
        "overlap": 0,
        "pcd0": str(pcd0_path),
        "pcd1": str(pcd1_path),
        "pcd_morphed": str(pcd_morphed_path),
        "gt_z_path": str(gt_z_path),
        "rotation": R,
        "translation": t,
        "scene_name": scene_name,
        "frag_id0": 0,
        "frag_id1": frag_id1,
        "kept_indices": kept_indices,
    }
    return metadata


In [56]:
FULL_PC_NAME =  "full_face.pth"
MORPHED_PC_NAME = "full_morphed_face.pth"

print(f"Computing average face from {len(all_files)} files...")
all_pts_list = []

for f in all_files:
    temp_pcd = o3d.io.read_point_cloud(str(f))
    all_pts_list.append(np.asarray(temp_pcd.points))

average_pts = np.mean(np.stack(all_pts_list), axis=0)

print(f"Average point cloud created. Shape: {average_pts.shape}")

Computing average face from 8000 files...
Average point cloud created. Shape: (10788, 3)


In [57]:
def process_file(file_path: Path, n_views=10, folder="train", save=True):
    pcd = o3d.io.read_point_cloud(str(file_path))
    pts = np.asarray(pcd.points)

    print(f"Processing file: {file_path.stem} with {pts.shape[0]} points. Data type: {pts.dtype}")
    print("Reference: Global Average")

    subject_path = DATASET_DIR / "data" / folder / file_path.stem

    file_idx = name_to_idx[file_path.name]
    sample_gt_z = all_gt_z[:, file_idx, :] 

    if save:
        subject_path.mkdir(parents=True, exist_ok=True)
        torch.save(average_pts, subject_path / FULL_PC_NAME)
        torch.save(pts.astype(np.float32), subject_path / MORPHED_PC_NAME)
        torch.save(sample_gt_z, subject_path / "gt_z.pth")

    metadata_list = []

    for i in range(n_views):
        transformed_points, R_inv, t_inv, _, kept_indices = generate_random_view(pts)

        metadata = build_metadata_dict(
            scene_name=file_path.stem,
            pcd0_path=(Path(folder) / file_path.stem / FULL_PC_NAME).as_posix(),
            pcd1_path=(Path(folder) / file_path.stem / f"view_{i+1}.pth").as_posix(),
            pcd_morphed_path= (Path(folder) / file_path.stem / MORPHED_PC_NAME).as_posix(),
            gt_z_path=(Path(folder) / file_path.stem / "gt_z.pth").as_posix(),
            R=R_inv,
            t=t_inv,
            frag_id1=i + 1,
            kept_indices=kept_indices
        )
        metadata_list.append(metadata)
        if save:    
            torch.save(transformed_points, subject_path / f"view_{i+1}.pth")

    return metadata_list

In [58]:
train_size = int(np.round(0.8 * len(random_sample)))
val_size = int(np.round(0.1 * len(random_sample)))
test_size = len(random_sample) - train_size - val_size

In [59]:
train_size, val_size, test_size

(2000, 250, 250)

In [60]:
train_metadata = []
for file_path in random_sample[:train_size]:
    metadata_list = process_file(file_path, n_views=2, folder="train")
    train_metadata.extend(metadata_list)
val_metadata = []
for file_path in random_sample[train_size:train_size+val_size]:
    metadata_list = process_file(file_path, n_views=2, folder="val")
    val_metadata.extend(metadata_list)

Processing file: 3478 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 8.01328992747696e-15
Reprojection error T_inv: 3.7776398860831214e-15
Processing file: 3890 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 9.145999150478842e-15
Reprojection error T_inv: 9.623763061693464e-15
Processing file: 2871 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 9.152440514592608e-15
Reprojection error T_inv: 9.153366351057232e-15
Processing file: 440 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 1.0773526089466666e-14
Reprojection error T_inv: 7.929161232345602e-15
Processing file: 5867 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 5.3296667960363994e-15
Reprojection error T_inv: 7.747389240967934e-15
Processing file: 4 with 10788 points. Data type: float64
Reference: Global Average
Reproj

In [61]:
test_metadata = []
for file_path in random_sample[train_size+val_size:]:
    metadata_list = process_file(file_path, n_views=1, folder="test")
    test_metadata.extend(metadata_list)

Processing file: 108 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 1.1538902799239405e-14
Processing file: 8828 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 9.824297613824598e-15
Processing file: 1472 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 6.781472283014893e-15
Processing file: 5571 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 1.456841390054829e-14
Processing file: 5964 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 7.45254265621338e-15
Processing file: 9787 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 6.789458596450796e-15
Processing file: 5155 with 10788 points. Data type: float64
Reference: Global Average
Reprojection error T_inv: 8.533383902374464e-15
Processing file: 7412 with 10788 points. Data type: float64
Ref

In [62]:
METADATA_DIR = DATASET_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

In [63]:
with open(METADATA_DIR / "train.pkl", "wb") as f:
    pickle.dump(train_metadata, f)

In [64]:
with open(METADATA_DIR / "val.pkl", "wb") as f:
    pickle.dump(val_metadata, f)

In [65]:
demo_folder = DATASET_DIR / "demo"
demo_folder.mkdir(parents=True, exist_ok=True)

In [66]:
for ex_id, example in enumerate(test_metadata):
    example_ref = torch.load(DATASET_DIR / "data" / example["pcd0"], weights_only=False)
    example_src = torch.load(DATASET_DIR / "data" / example["pcd1"], weights_only=False)
    example_gt_morphed = torch.load(DATASET_DIR / "data" / example["pcd_morphed"], weights_only=False)
    np.save(demo_folder / f"ref_{ex_id}.npy", example_ref)
    np.save(demo_folder / f"src_{ex_id}.npy", example_src)
    np.save(demo_folder / f"morphed_full_{ex_id}.npy", example_gt_morphed)
    rot = example["rotation"]
    t = example["translation"]
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = rot
    T[:3, 3] = t
    np.save(demo_folder / f"gt_{ex_id}.npy", T)


### Inspect Test Data

In [67]:
#test_sample = test_metadata[np.random.randint(len(test_metadata))]
test_sample = test_metadata[0]

In [68]:
test_sample

{'overlap': 0,
 'pcd0': 'test/108/full_face.pth',
 'pcd1': 'test/108/view_1.pth',
 'pcd_morphed': 'test/108/full_morphed_face.pth',
 'gt_z_path': 'test/108/gt_z.pth',
 'rotation': array([[ 0.46402371,  0.14376172,  0.87407927],
        [-0.85110908,  0.34588228,  0.39494149],
        [-0.24555106, -0.92719902,  0.28285448]]),
 'translation': array([ 0.73854324,  0.94515215, -0.18239509]),
 'scene_name': '108',
 'frag_id0': 0,
 'frag_id1': 1,
 'kept_indices': array([4545, 1155, 9433, ..., 6877, 9799, 1342])}

In [69]:
test_src = torch.load(DATASET_DIR / "data" / test_sample["pcd1"], weights_only=False)
test_ref = torch.load(DATASET_DIR / "data" / test_sample["pcd0"], weights_only=False)

In [70]:
file_path = Path("C:/Eli Folder temp/geotransformer-faces-updated/data/UHM_downsampled/train") / f"{test_sample['scene_name']}.ply" 

In [71]:
ref_original = pcd = o3d.io.read_point_cloud(str(file_path))

In [72]:
test_ref.shape

(10788, 3)

In [73]:
np.asarray(ref_original.points).shape

(10788, 3)

In [74]:
np.mean(np.linalg.norm(np.asarray(ref_original.points) - test_ref, axis=1))

np.float64(0.03538939614261289)

In [75]:
src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0])
ref_pcd = o3d.geometry.PointCloud()
ref_pcd.points = o3d.utility.Vector3dVector(test_ref)
ref_pcd.paint_uniform_color([0.0, 1.0, 0.0])


PointCloud with 10788 points.

In [76]:
o3d.visualization.draw_plotly([ref_pcd, src_pcd])

In [77]:
t = test_sample["translation"]
rot = test_sample["rotation"]
T = np.eye(4, dtype=np.float64)
T[:3, :3] = rot
T[:3, 3] = t

In [78]:
# ref is mean face here so alignment won't look perfect
o3d.visualization.draw_plotly([ref_pcd, src_pcd.transform(T)])

In [79]:
# here we set ref to the full morphed sample of src to see perfect algignment

test_gt_morphed = torch.load(DATASET_DIR / "data" / test_sample["pcd_morphed"], weights_only=False)

morphed_pcd = o3d.geometry.PointCloud()
morphed_pcd.points = o3d.utility.Vector3dVector(test_gt_morphed)
morphed_pcd.paint_uniform_color([0.0, 1.0, 0.0]) 

src_pcd = o3d.geometry.PointCloud()
src_pcd.points = o3d.utility.Vector3dVector(test_src)
src_pcd.paint_uniform_color([1.0, 0.0, 0.0]) 

o3d.visualization.draw_plotly([morphed_pcd, src_pcd.transform(T)])

In [80]:
kept_indices = test_sample["kept_indices"]

In [81]:
r_errors = (np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices]

In [82]:
np.linalg.norm(r_errors, axis=1).max()

np.float64(0.09466578220707007)

In [83]:
print(f"Reproyection error T_inv {np.linalg.norm((np.hstack((test_src, np.ones((test_src.shape[0], 1)))) @ T.T)[:, :3] - test_ref[kept_indices])}")


Reproyection error T_inv 1.961280061183855


In [84]:
error = np.asarray(ref_pcd.points)[kept_indices] - np.asarray(src_pcd.points)

In [85]:
np.linalg.norm(error, axis=1).mean()

np.float64(0.032975276771163874)

In [86]:
np.sum(np.linalg.norm(error, axis=1))

np.float64(94.375242119071)